# Araseの粒子データのplot、解析

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# LEP-e

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import numpy as np

time_range = ['20220901/21:00:00', '20220902/00:00:00']

ergpy.lepe(trange=time_range, datatype='3dflux', level='l2')

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
import xarray as xr
import numpy as np

ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True)

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.erg_mgf_spintone_rm as emsr
import importlib
importlib.reload(emsr)

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

background_time_sec = 100 #[sec]

time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

In [ ]:
psp.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})
ergpy.orb(trange=time_range, level='l2', datatype='def')

In [ ]:
energy_list = psp.get_data('erg_lepe_l2_3dflux_FEDU', xarray=True).v1[0, :].data

energy_list = np.unique(np.sort(energy_list))

print(energy_list)

In [ ]:
#for i, energy in enumerate(energy_list):
#    psp.projects.erg.erg_lep_part_products(
#        'erg_lepe_l2_3dflux_FEDU',
#        outputs=['pa'],
#        energy=[np.trunc(energy), np.ceil(energy)],
#        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
#        pos_name='erg_orb_l2_pos_gse',
#        suffix='_'+str(i)
#    )

In [ ]:
#import numpy as np
#import xarray as xr
#import pandas as pd
#
#energy = np.asarray(energy_list, dtype=float)
#
#flux_list = []
#pa_ref = None
#
#for i, ene in enumerate(energy):
#    varname = f"erg_lepe_l2_3dflux_FEDU_pa_{i}"
#    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))
#
#    if da is None:
#        raise ValueError(f"{varname} が取得できない")
#
#    # flux: (time, v_dim) -> (time, energy, v_dim)
#    da_flux = da.expand_dims(energy=[ene])
#
#    flux_list.append(da_flux)
#
#    # pitch angle bin を確認
#    pa_now = da["spec_bins"].values   # shape=(time, v_dim)
#
#    if pa_ref is None:
#        pa_ref = pa_now
#    else:
#        if not np.allclose(pa_now, pa_ref, equal_nan=True):
#            raise ValueError(
#                f"{varname} の spec_bins が他の energy channel と一致しない"
#            )
#
## concat 後、(energy, time, v_dim) なので並べ替え
#da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")
#
## pitch angle が time に依らず一定か確認
#if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
#    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")
#
#pitch_angle = pa_ref[0, :]   # shape=(v_dim,)
#
## DataArray 化
#da_flux_3d = xr.DataArray(
#    da_flux_all.values,
#    dims=("time", "energy", "pitch_angle"),
#    coords={
#        "time": da_flux_all["time"].values,
#        "energy": energy,
#        "pitch_angle": pitch_angle,
#    },
#    name="FEDU_flux",
#    attrs={
#        "units": "#/s/cm^2/sr/eV",
#        "description": "LEP-e 3dflux as a function of time, energy, and pitch angle",
#    }
#)
#
#print(da_flux_3d)
#
#from pathlib import Path
#import pandas as pd
#
#t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
#t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")
#
#save_path = Path(
#    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPe_da_pa_energy_{t0_str}_{t1_str}.nc"
#)
#
#da_flux_3d.to_netcdf(save_path)
#print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPe_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPe_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPe = xr.open_dataset(LEPe_flux_Path)

print(da_LEPe)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import pandas as pd

#def plot_lepe_pitchangle_polar(
#    da_LEPe,
#    time,
#    ylabel,
#    vmin=1e2,
#    vmax=1e5,
#    emin=1e1,
#    emax=1e4,
#    cmap="turbo",
#    figsize=(6, 6),
#):
#    time = pd.Timestamp(time)
#
#    da_plot = (
#        da_LEPe["FEDU_flux"]
#        .sel(time=time, method="nearest")
#        .transpose("energy", "pitch_angle")
#    )
#
#    time_nearest = pd.Timestamp(da_plot.time.values).round("s")
#    time_nearest_next = time_nearest + pd.Timedelta(seconds=8)
#
#    E = da_plot["energy"].values
#    alpha_deg = da_plot["pitch_angle"].values
#    flux = da_plot.values.astype(float)
#
#    flux[~np.isfinite(flux)] = np.nan
#    flux[flux <= 0] = np.nan
#
#    def make_edges(x):
#        x = np.asarray(x, dtype=float)
#        dx = np.diff(x)
#        x_edge = np.empty(x.size + 1, dtype=float)
#        x_edge[1:-1] = 0.5 * (x[:-1] + x[1:])
#        x_edge[0] = x[0] - 0.5 * dx[0]
#        x_edge[-1] = x[-1] + 0.5 * dx[-1]
#        return x_edge
#
#    E_edge = make_edges(E)
#    alpha_edge = np.deg2rad(make_edges(alpha_deg))
#
#    TT, RR = np.meshgrid(alpha_edge, E_edge, indexing="xy")
#
#    mpl.rcParams["font.size"] = 20
#
#    fig = plt.figure(figsize=figsize)
#    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.03], wspace=0)
#
#    ax_0 = fig.add_subplot(gs[0, 0], projection="polar")
#    cax_0 = fig.add_subplot(gs[0, 1])
#
#    ax_0.set_theta_zero_location("N")
#    ax_0.set_theta_direction(-1)
#    ax_0.set_thetamin(0)
#    ax_0.set_thetamax(180)
#
#    ax_0.set_rscale("log")
#    ax_0.set_rlabel_position(185)
#    ax_0.set_ylim(emin, emax)
#    ax_0.set_ylabel(ylabel, labelpad=-30)
#
#    mesh = ax_0.pcolormesh(
#        TT,
#        RR,
#        flux,
#        cmap=cmap,
#        norm=mcolors.LogNorm(vmin=vmin, vmax=vmax),
#        shading="flat",
#    )
#
#    title_str = f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}"
#    ax_0.set_title(title_str, pad=20)
#
#    cb = fig.colorbar(mesh, cax=cax_0)
#    cb.set_label(r'[$\mathrm{s}^{-1} \mathrm{cm}^{-2} \mathrm{str}^{-1} \mathrm{eV}^{-1}$]')
#
#    ax_0.minorticks_on()
#    ax_0.set_thetagrids(np.rad2deg(np.linspace(0, np.pi, 7)))
#    ax_0.grid(True, which="both", linestyle=":", alpha=0.5)
#    ax_0.grid(True, which="major", linestyle="solid", alpha=1, c="k")
#    ax_0.tick_params(axis="x", pad=10)
#
#    fig.subplots_adjust(left=0.11, right=0.87)
#
#    return fig, ax_0, cax_0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import pandas as pd

def plot_lepe_pitchangle_polar(
    da_LEPe,
    time,
    ylabel,
    vmin=1e2,
    vmax=1e5,
    emin=1e1,
    emax=1e4,
    cmap="turbo",
    figsize=(6, 6),
    pa_range_for_stats=(10, 170),   # 統計に使うpitch angle範囲
    overlay_peak_line=True,
):
    time = pd.Timestamp(time)

    da_plot = (
        da_LEPe["FEDU_flux"]
        .sel(time=time, method="nearest")
        .transpose("energy", "pitch_angle")
    )

    time_nearest = pd.Timestamp(da_plot.time.values).round("s")
    time_nearest_next = time_nearest + pd.Timedelta(seconds=8)

    E = da_plot["energy"].values                  # shape: (nE,)
    alpha_deg = da_plot["pitch_angle"].values     # shape: (nPA,)
    flux = da_plot.values.astype(float)           # shape: (nE, nPA)

    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan

    def make_edges(x):
        x = np.asarray(x, dtype=float)
        if x.size < 2:
            raise ValueError("Need at least 2 points to make edges.")
        dx = np.diff(x)
        x_edge = np.empty(x.size + 1, dtype=float)
        x_edge[1:-1] = 0.5 * (x[:-1] + x[1:])
        x_edge[0] = x[0] - 0.5 * dx[0]
        x_edge[-1] = x[-1] + 0.5 * dx[-1]
        return x_edge

    E_edge = make_edges(E)
    alpha_edge = np.deg2rad(make_edges(alpha_deg))
    TT, RR = np.meshgrid(alpha_edge, E_edge, indexing="xy")

    mpl.rcParams["font.size"] = 20

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.03], wspace=0)

    ax_0 = fig.add_subplot(gs[0, 0], projection="polar")
    cax_0 = fig.add_subplot(gs[0, 1])

    ax_0.set_theta_zero_location("N")
    ax_0.set_theta_direction(-1)
    ax_0.set_thetamin(0)
    ax_0.set_thetamax(180)

    ax_0.set_rscale("log")
    ax_0.set_rlabel_position(185)
    ax_0.set_ylim(emin, emax)
    ax_0.set_ylabel(ylabel, labelpad=-30)

    mesh = ax_0.pcolormesh(
        TT,
        RR,
        flux,
        cmap=cmap,
        norm=mcolors.LogNorm(vmin=vmin, vmax=vmax),
        shading="flat",
    )

    # -----------------------------
    # 各 pitch angle におけるピーク energy を求める
    # -----------------------------
    peak_energy = np.full(alpha_deg.shape, np.nan, dtype=float)
    peak_flux = np.full(alpha_deg.shape, np.nan, dtype=float)

    for j in range(len(alpha_deg)):
        col = flux[:, j]
        valid = np.isfinite(col) & np.isfinite(E) & (E >= emin) & (E <= emax)
        if np.any(valid):
            idx_local = np.nanargmax(col[valid])
            E_valid = E[valid]
            F_valid = col[valid]
            peak_energy[j] = E_valid[idx_local]
            peak_flux[j] = F_valid[idx_local]

    # -----------------------------
    # 線を重ねる
    # -----------------------------
    if overlay_peak_line:
        valid_line = np.isfinite(peak_energy)
        ax_0.plot(
            np.deg2rad(alpha_deg[valid_line]),
            peak_energy[valid_line],
            color="k",
            lw=2.0,
            marker="o",
            ms=3,
            zorder=5,
        )

    # -----------------------------
    # 代表値を算出
    # -----------------------------
    pa_min, pa_max = pa_range_for_stats
    valid_stats = (
        np.isfinite(peak_energy)
        & np.isfinite(peak_flux)
        & (alpha_deg >= pa_min)
        & (alpha_deg <= pa_max)
    )

    # 単純平均より、ピークflux重み付き平均の方が ring の主成分を反映しやすい
    if np.any(valid_stats):
        weights = peak_flux[valid_stats].copy()
        weights = np.where(np.isfinite(weights) & (weights > 0), weights, 0.0)

        if np.sum(weights) > 0:
            E_mean = np.sum(weights * peak_energy[valid_stats]) / np.sum(weights)
            E_std = np.sqrt(
                np.sum(weights * (peak_energy[valid_stats] - E_mean) ** 2) / np.sum(weights)
            )
        else:
            E_mean = np.nanmean(peak_energy[valid_stats])
            E_std = np.nanstd(peak_energy[valid_stats])
    else:
        E_mean = np.nan
        E_std = np.nan

    # -----------------------------
    # title
    # -----------------------------
    title_str = (
        f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}\n"
        f"{E_mean:.2f} ± {E_std:.2f} eV"
        if np.isfinite(E_mean)
        else f"{time_nearest.strftime('%H:%M:%S')} - {time_nearest_next.strftime('%H:%M:%S')}"
    )
    ax_0.set_title(title_str, pad=20)

    cb = fig.colorbar(mesh, cax=cax_0)
    cb.set_label(r'[$\mathrm{s}^{-1} \mathrm{cm}^{-2} \mathrm{str}^{-1} \mathrm{eV}^{-1}$]')

    ax_0.minorticks_on()
    ax_0.set_thetagrids(np.rad2deg(np.linspace(0, np.pi, 7)))
    ax_0.grid(True, which="both", linestyle=":", alpha=0.5)
    ax_0.grid(True, which="major", linestyle="solid", alpha=1, c="k")
    ax_0.tick_params(axis="x", pad=10)

    fig.subplots_adjust(left=0.11, right=0.87)

    results = {
        "time_nearest": time_nearest,
        "time_nearest_next": time_nearest_next,
        "pitch_angle_deg": alpha_deg,
        "peak_energy_eV": peak_energy,
        "peak_flux": peak_flux,
        "E_ring_mean_eV": E_mean,
        "E_ring_std_eV": E_std,
    }

    return fig, ax_0, cax_0, results

In [ ]:
fig, ax, cax, results = plot_lepe_pitchangle_polar(
    da_LEPe=da_LEPe,
    time="2022-09-01T22:37:00",
    ylabel=r"$\mathrm{e}^{-}$ Energy [eV]",
    pa_range_for_stats=(10, 170),
)
plt.show()
print("Representative ring energy:")
print(f"{results['E_ring_mean_eV']:.1f} ± {results['E_ring_std_eV']:.1f} eV")


In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

ylabel_lepe = r"$\mathrm{e}^{-}$ Energy [eV]"

out_dir = f'/mnt/j/KAW_observation/LEP-e_pitch_angle_each_time/20220901/21-24_energy_pa'
os.makedirs(out_dir, exist_ok=True)

out_dir_pdf = f'{out_dir}/PDF/'
os.makedirs(out_dir_pdf, exist_ok=True)
out_dir_png = f'{out_dir}/PNG/'
os.makedirs(out_dir_png, exist_ok=True)

csv_path = f'{out_dir}/ring_energy_summary.csv'

# 毎回全配列に percentiles を掛けると無駄なので先に一度だけ計算
flux_all = np.where(da_LEPe.FEDU_flux.values > 0, da_LEPe.FEDU_flux.values, np.nan)
vmin_global = np.nanpercentile(flux_all, 50)
vmax_global = np.nanpercentile(flux_all, 99)

def process_and_save_plot(t):
    ts = pd.Timestamp(t)

    fig = None
    try:
        fig, ax, cax, results = plot_lepe_pitchangle_polar(
            da_LEPe=da_LEPe,
            time=ts,
            ylabel=ylabel_lepe,
            vmin=vmin_global,
            vmax=vmax_global,
            pa_range_for_stats=(10, 170),
        )

        base = ts.strftime('%Y-%m-%dT%H%M%S')
        png_path = os.path.join(out_dir_png, base + '.png')
        pdf_path = os.path.join(out_dir_pdf, base + '.pdf')

        fig.savefig(png_path, dpi=200, bbox_inches='tight')
        fig.savefig(pdf_path, bbox_inches='tight')

        # CSV 用の1行
        row = {
            "time_input": ts.isoformat(),
            "time_nearest": pd.Timestamp(results["time_nearest"]).isoformat(),
            "time_nearest_next": pd.Timestamp(results["time_nearest_next"]).isoformat(),
            "E_ring_mean_eV": results["E_ring_mean_eV"],
            "E_ring_std_eV": results["E_ring_std_eV"],
        }

        # pitch-angle ごとの peak energy も保存したいなら文字列化して持たせる
        pa = np.asarray(results["pitch_angle_deg"], dtype=float)
        epeak = np.asarray(results["peak_energy_eV"], dtype=float)
        pflux = np.asarray(results["peak_flux"], dtype=float)

        row["pitch_angle_deg"] = ",".join(
            "nan" if not np.isfinite(x) else f"{x:.1f}" for x in pa
        )
        row["peak_energy_eV_by_pa"] = ",".join(
            "nan" if not np.isfinite(x) else f"{x:.3f}" for x in epeak
        )
        row["peak_flux_by_pa"] = ",".join(
            "nan" if not np.isfinite(x) else f"{x:.6g}" for x in pflux
        )

        return row

    except Exception as e:
        # 並列処理ではどの時刻で落ちたか分かるようにして返す
        return {
            "time_input": ts.isoformat(),
            "time_nearest": None,
            "time_nearest_next": None,
            "E_ring_mean_eV": np.nan,
            "E_ring_std_eV": np.nan,
            "pitch_angle_deg": None,
            "peak_energy_eV_by_pa": None,
            "peak_flux_by_pa": None,
            "error": repr(e),
        }

    finally:
        if fig is not None:
            plt.close(fig)

time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
t_min, t_max = pd.to_datetime(time_range)
time_grid = da_LEPe.sel(time=slice(t_min, t_max)).time.values

with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame"):
    results_list = Parallel(
        n_jobs=os.cpu_count(),
        backend="loky",
        verbose=0
    )(
        delayed(process_and_save_plot)(t) for t in time_grid
    )

df_results = pd.DataFrame(results_list)
df_results = df_results.sort_values("time_input").reset_index(drop=True)
df_results.to_csv(csv_path, index=False)

print(f"Finished saving all plots and CSV: {csv_path}")

# LEP-i

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import numpy as np

ergpy.lepi(trange=time_range, datatype='3dflux', level='l2')

# $\mathrm{H}^{+}$

In [ ]:
energy_list_P = psp.get_data('erg_lepi_l2_3dflux_FPDU', xarray=True).v1 * 1E3

energy_list_P = np.unique(np.sort(energy_list_P))

print(energy_list_P)

In [ ]:
for i, energy in enumerate(energy_list_P):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FPDU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_P, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FPDU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FPDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FPDU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FPDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_P = xr.open_dataset(LEPi_FPDU_flux_Path)

print(da_LEPi_P)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#ylabel_lepi_P = r"$\mathrm{H}^{+}$ Energy [eV]"
#
#out_dir = f'/mnt/j/KAW_observation/LEP-i_P_pitch_angle_each_time/20220901/21-24_energy_pa'
#os.makedirs(out_dir, exist_ok=True)
#
#out_dir_pdf = f'{out_dir}/PDF/'
#os.makedirs(out_dir_pdf, exist_ok=True)
#out_dir_png = f'{out_dir}/PNG/'
#os.makedirs(out_dir_png, exist_ok=True)
#
#def process_and_save_plot(t):
#    fig, ax, cax = plot_lepe_pitchangle_polar(
#        da_LEPe=da_LEPi_P,
#        time=t,
#        ylabel=ylabel_lepi_P,
#        vmin=np.nanpercentile(np.where(da_LEPi_P.FEDU_flux.values > 0, da_LEPi_P.FEDU_flux.values, np.nan), 50),
#        vmax=np.nanpercentile(np.where(da_LEPi_P.FEDU_flux.values > 0, da_LEPi_P.FEDU_flux.values, np.nan), 99)
#    )
#    if fig is None:
#        return None
#
#    ts = pd.Timestamp(t)
#    base = ts.strftime('%Y-%m-%dT%H%M%S')
#    png_path = os.path.join(out_dir_png, base + '.png')
#    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')
#
#    try:
#        fig.savefig(png_path, dpi=200, bbox_inches='tight')
#        fig.savefig(pdf_path, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    return
#
#time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
#
#t_min, t_max    = pd.to_datetime(time_range)
#time_grid = da_LEPi_P.sel(time=slice(t_min, t_max)).time.values
#
#with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
#    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
#        delayed(process_and_save_plot)(t) for t in time_grid
#    )
#print('Finished saving all plots!')

# $\mathrm{He}^{+}$

In [ ]:
energy_list_HE = psp.get_data('erg_lepi_l2_3dflux_FHEDU', xarray=True).v1 * 1E3

energy_list_HE = np.unique(np.sort(energy_list_HE))

print(energy_list_HE)

In [ ]:
for i, energy in enumerate(energy_list_HE):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FHEDU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_HE, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FHEDU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FHEDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FHEDU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FHEDU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_HE = xr.open_dataset(LEPi_FHEDU_flux_Path)

print(da_LEPi_HE)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#ylabel_lepi_HE = r"$\mathrm{He}^{+}$ Energy [eV]"
#
#out_dir = f'/mnt/j/KAW_observation/LEP-i_HE_pitch_angle_each_time/20220901/21-24_energy_pa'
#os.makedirs(out_dir, exist_ok=True)
#
#out_dir_pdf = f'{out_dir}/PDF/'
#os.makedirs(out_dir_pdf, exist_ok=True)
#out_dir_png = f'{out_dir}/PNG/'
#os.makedirs(out_dir_png, exist_ok=True)
#
#def process_and_save_plot(t):
#    fig, ax, cax = plot_lepe_pitchangle_polar(
#        da_LEPe=da_LEPi_HE,
#        time=t,
#        ylabel=ylabel_lepi_HE,
#        vmin=np.nanpercentile(np.where(da_LEPi_HE.FEDU_flux.values > 0, da_LEPi_HE.FEDU_flux.values, np.nan), 50),
#        vmax=np.nanpercentile(np.where(da_LEPi_HE.FEDU_flux.values > 0, da_LEPi_HE.FEDU_flux.values, np.nan), 99)
#    )
#    if fig is None:
#        return None
#
#    ts = pd.Timestamp(t)
#    base = ts.strftime('%Y-%m-%dT%H%M%S')
#    png_path = os.path.join(out_dir_png, base + '.png')
#    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')
#
#    try:
#        fig.savefig(png_path, dpi=200, bbox_inches='tight')
#        fig.savefig(pdf_path, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    return
#
#time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
#
#t_min, t_max    = pd.to_datetime(time_range)
#time_grid = da_LEPi_HE.sel(time=slice(t_min, t_max)).time.values
#
#with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
#    results_list = Parallel(n_jobs=os.cpu_count(), backend="loky", verbose=0)(
#        delayed(process_and_save_plot)(t) for t in time_grid
#    )
#print('Finished saving all plots!')

# $\mathrm{O}^{+}$

In [ ]:
energy_list_O = psp.get_data('erg_lepi_l2_3dflux_FODU', xarray=True).v1 * 1E3

energy_list_O = np.unique(np.sort(energy_list_O))

print(energy_list_O)

In [ ]:
for i, energy in enumerate(energy_list_O):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FODU',
        outputs=['pa'],
        energy=[np.trunc(energy), np.ceil(energy)],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_gse',
        suffix='_'+str(i)
    )

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

energy = np.asarray(energy_list_O, dtype=float)

flux_list = []
pa_ref = None

for i, ene in enumerate(energy):
    varname = f"erg_lepi_l2_3dflux_FODU_pa_{i}"
    da = psp.get_data(varname, xarray=True).sel(time=slice(*time_range))

    if da is None:
        raise ValueError(f"{varname} が取得できない")

    # flux: (time, v_dim) -> (time, energy, v_dim)
    da_flux = da.expand_dims(energy=[ene])

    flux_list.append(da_flux)

    # pitch angle bin を確認
    pa_now = da["spec_bins"].values   # shape=(time, v_dim)

    if pa_ref is None:
        pa_ref = pa_now
    else:
        if not np.allclose(pa_now, pa_ref, equal_nan=True):
            raise ValueError(
                f"{varname} の spec_bins が他の energy channel と一致しない"
            )

# concat 後、(energy, time, v_dim) なので並べ替え
da_flux_all = xr.concat(flux_list, dim="energy").transpose("time", "energy", "v_dim")

# pitch angle が time に依らず一定か確認
if not np.allclose(pa_ref, pa_ref[0, :][None, :], equal_nan=True):
    raise ValueError("spec_bins が time に依存しているため、1次元 pitch_angle 座標にできない")

pitch_angle = pa_ref[0, :]   # shape=(v_dim,)

# DataArray 化
da_flux_3d = xr.DataArray(
    da_flux_all.values,
    dims=("time", "energy", "pitch_angle"),
    coords={
        "time": da_flux_all["time"].values,
        "energy": energy,
        "pitch_angle": pitch_angle,
    },
    name="FEDU_flux",
    attrs={
        "units": "#/s/cm^2/sr/eV",
        "description": "LEP-i 3dflux as a function of time, energy, and pitch angle",
    }
)

print(da_flux_3d)

from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

save_path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FODU_da_pa_energy_{t0_str}_{t1_str}.nc"
)

da_flux_3d.to_netcdf(save_path)
print(save_path)

In [ ]:
import xarray as xr
from pathlib import Path
import pandas as pd

t0_str = pd.to_datetime(time_range[0]).strftime("%Y%m%d_%H%M%S")
t1_str = pd.to_datetime(time_range[1]).strftime("%Y%m%d_%H%M%S")

LEPi_FODU_flux_Path = Path(
    f"/mnt/j/observation_data/Arase_analysis_save_data/LEPi_FODU_da_pa_energy_{t0_str}_{t1_str}.nc"
)
da_LEPi_O = xr.open_dataset(LEPi_FODU_flux_Path)

print(da_LEPi_O)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#ylabel_lepi_O = r"$\mathrm{O}^{+}$ Energy [eV]"
#
#out_dir = f'/mnt/j/KAW_observation/LEP-i_O_pitch_angle_each_time/20220901/21-24_energy_pa'
#os.makedirs(out_dir, exist_ok=True)
#
#out_dir_pdf = f'{out_dir}/PDF/'
#os.makedirs(out_dir_pdf, exist_ok=True)
#out_dir_png = f'{out_dir}/PNG/'
#os.makedirs(out_dir_png, exist_ok=True)
#
#def process_and_save_plot(t):
#    fig, ax, cax = plot_lepe_pitchangle_polar(
#        da_LEPe=da_LEPi_O,
#        time=t,
#        ylabel=ylabel_lepi_O,
#        vmin=np.nanpercentile(np.where(da_LEPi_O.FEDU_flux.values > 0, da_LEPi_O.FEDU_flux.values, np.nan), 50),
#        vmax=np.nanpercentile(np.where(da_LEPi_O.FEDU_flux.values > 0, da_LEPi_O.FEDU_flux.values, np.nan), 99)
#    )
#    if fig is None:
#        return None
#
#    ts = pd.Timestamp(t)
#    base = ts.strftime('%Y-%m-%dT%H%M%S')
#    png_path = os.path.join(out_dir_png, base + '.png')
#    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')
#
#    try:
#        fig.savefig(png_path, dpi=200, bbox_inches='tight')
#        fig.savefig(pdf_path, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    return
#
#time_range = ['2022-09-01T21:00:00', '2022-09-02T00:00:00']
#
#t_min, t_max    = pd.to_datetime(time_range)
#time_grid = da_LEPi_O.sel(time=slice(t_min, t_max)).time.values
#
#with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
#    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
#        delayed(process_and_save_plot)(t) for t in time_grid
#    )
#print('Finished saving all plots!')